#### Pytorch: 
- Temporal Fusion Transformers
- attention mechanism
- Graph Neural Networks (GNNs)
- N-BEATS
- DeepAR
#### ML-Flow
#### Fast API

### Importing libraries

In [1]:
import mlflow
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn
from torch.cuda import CUDAGraph
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [2]:
is_cuda_available = torch.cuda.is_available()

In [3]:
mlflow.set_experiment("Global_Weather_Temperature_Prediction")

<Experiment: artifact_location='file:c:/Users/Shant/OneDrive/Desktop/Data_Science/Coding/Final-Project/mlruns/1', creation_time=1776603583505, experiment_id='1', last_update_time=1776603583505, lifecycle_stage='active', name='Global_Weather_Temperature_Prediction', tags={}, trace_location=None, workspace='default'>

In [4]:
df = pd.read_csv(
    r"C:\Users\Shant\OneDrive\Documents\My Folder\Dataset\GlobalWeatherRepository.csv")
df.head()

,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,...,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,...,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,...,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45,6.3,43.3,Light drizzle,...,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45,26.0,78.8,Partly cloudy,...,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55


In [5]:
df.columns

Index(['country', 'location_name', 'latitude', 'longitude', 'timezone',
       'last_updated_epoch', 'last_updated', 'temperature_celsius',
       'temperature_fahrenheit', 'condition_text', 'wind_mph', 'wind_kph',
       'wind_degree', 'wind_direction', 'pressure_mb', 'pressure_in',
       'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius',
       'feels_like_fahrenheit', 'visibility_km', 'visibility_miles',
       'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide',
       'air_quality_Ozone', 'air_quality_Nitrogen_dioxide',
       'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10',
       'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise',
       'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination'],
      dtype='object')

In [6]:
df.describe()

,latitude,longitude,last_updated_epoch,temperature_celsius,temperature_fahrenheit,wind_mph,wind_kph,wind_degree,pressure_mb,pressure_in,...,gust_kph,air_quality_Carbon_Monoxide,air_quality_Ozone,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,moon_illumination
count,136438.000000,136438.000000,1.364380e+05,136438.000000,136438.000000,136438.000000,136438.000000,136438.000000,136438.000000,136438.000000,...,136438.000000,136438.000000,136438.000000,136438.000000,136438.000000,136438.000000,136438.000000,136438.000000,136438.000000,136438.000000
mean,19.210920,21.957523,1.746212e+09,21.270182,70.288099,8.012285,12.898154,168.812691,1014.065722,29.944747,...,18.243974,459.481919,58.009895,14.979770,10.322060,24.157571,48.299958,1.697702,2.601885,49.566800
std,24.416116,65.787384,1.752327e+07,9.688032,17.438311,7.204160,11.590787,103.740755,10.333007,0.305087,...,13.796383,756.132716,30.720288,23.660363,35.529098,36.776617,148.890044,0.939427,2.440784,35.106112
min,-41.300000,-175.200000,1.715849e+09,-29.800000,-21.600000,2.200000,3.600000,1.000000,947.000000,27.960000,...,3.600000,-9999.000000,0.000000,0.000000,-9999.000000,0.168000,-1848.150000,1.000000,1.000000,0.000000
25%,4.050300,-6.836100,1.731056e+09,16.000000,60.700000,3.800000,6.100000,80.000000,1010.000000,29.830000,...,10.200000,201.900000,38.000000,1.678000,1.110000,7.050000,9.990000,1.000000,1.000000,15.000000
50%,17.250000,23.236100,1.746262e+09,24.000000,75.100000,6.700000,10.800000,161.000000,1014.000000,29.930000,...,15.300000,293.700000,55.000000,5.638000,2.405000,14.150000,19.980000,1.000000,2.000000,49.000000
75%,40.400000,49.882200,1.761377e+09,28.000000,82.400000,11.000000,17.600000,256.000000,1018.000000,30.060000,...,24.200000,458.850000,74.000000,17.390000,8.325000,27.750000,41.440000,2.000000,3.000000,84.000000
max,64.150000,179.220000,1.776579e+09,49.200000,120.600000,1841.200000,2963.200000,360.000000,3006.000000,88.770000,...,2970.400000,38879.398000,480.700000,427.700000,521.330000,1614.100000,6037.290000,6.000000,10.000000,100.000000


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 136438 entries, 0 to 136437
Data columns (total 41 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   country                       136438 non-null  object 
 1   location_name                 136438 non-null  object 
 2   latitude                      136438 non-null  float64
 3   longitude                     136438 non-null  float64
 4   timezone                      136438 non-null  object 
 5   last_updated_epoch            136438 non-null  int64  
 6   last_updated                  136438 non-null  object 
 7   temperature_celsius           136438 non-null  float64
 8   temperature_fahrenheit        136438 non-null  float64
 9   condition_text                136438 non-null  object 
 10  wind_mph                      136438 non-null  float64
 11  wind_kph                      136438 non-null  float64
 12  wind_degree                   136438 non-nul

In [8]:
df.shape

(136438, 41)

### Preprocessing 

In [9]:
cols_to_drop = [
    'temperature_fahrenheit', 'feels_like_fahrenheit',
    'wind_mph', 'pressure_in', 'precip_in',
    'visibility_miles', 'gust_mph', 'last_updated_epoch'
]
df_cleaned = df.drop(columns=cols_to_drop)

In [10]:
df_cleaned['last_updated'] = pd.to_datetime(df_cleaned['last_updated'])
df_cleaned['year'] = df_cleaned['last_updated'].dt.year
df_cleaned['month'] = df_cleaned['last_updated'].dt.month
df_cleaned['day'] = df_cleaned['last_updated'].dt.day
df_cleaned['hour'] = df_cleaned['last_updated'].dt.hour

In [11]:
# Drop the original timestamp string now that we have the numerical parts
df_cleaned = df_cleaned.drop(columns=['last_updated'])

In [12]:
# 4. Handle categorical columns using Label Encoding
categorical_cols = [
    'country', 'location_name', 'timezone', 'condition_text',
    'wind_direction', 'sunrise', 'sunset', 'moonrise', 'moonset', 'moon_phase'
]

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    # Convert to string first to ensure the encoder can process it cleanly
    df_cleaned[col] = le.fit_transform(df_cleaned[col].astype(str))
    label_encoders[col] = le

In [13]:
df_cleaned.head()

,country,location_name,latitude,longitude,timezone,temperature_celsius,condition_text,wind_kph,wind_degree,wind_direction,...,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination,year,month,day,hour
0,0,102,34.52,69.18,108,26.6,31,13.3,338,6,...,113,243,1347,24,7,55,2024,5,16,13
1,1,233,41.33,19.82,174,19.0,32,11.2,320,7,...,144,307,1439,150,7,55,2024,5,16,10
2,2,13,36.76,3.05,3,23.0,45,15.1,280,13,...,163,303,33,150,7,55,2024,5,16,9
3,3,16,42.50,1.52,139,6.3,11,11.9,215,12,...,214,384,147,304,7,55,2024,5,16,10
4,4,126,-8.84,13.23,29,26.0,32,13.0,150,10,...,195,188,37,1398,7,55,2024,5,16,9


### Model building

In [14]:
X = df_cleaned.drop(columns=['temperature_celsius', 'feels_like_celsius'])
y = df_cleaned['temperature_celsius'].values

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, shuffle=True)

In [16]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [17]:
# class WeatherDataset(Dataset):
#     def __init__(self, X, y):
#         # Convert data to PyTorch Tensors
#         self.X = torch.tensor(X, dtype=torch.float32)
#         self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

#     def __len__(self):
#         return len(self.X)

#     def __getitem__(self, idx):
#         return self.X[idx], self.y[idx]

# explain why here using magic methods

In [18]:
from torch.utils.data import TensorDataset, DataLoader
import torch


X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)


X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [19]:
# Check for NVIDIA GPU (cuda), Apple Silicon (mps), or fallback to CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cuda


In [ ]:
class WeatherPredictor(nn.Module):
    def __init__(self, input_dim):
        super(WeatherPredictor, self).__init__()
        self.layer1 = nn.Linear(input_dim, 64)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.output_layer = nn.Linear(32, 1)

    def forward(self, x):
        x = self.relu1(self.layer1(x))
        x = self.relu2(self.layer2(x))
        x = self.output_layer(x)
        return x


input_dim = X_train_scaled.shape[1]
model = WeatherPredictor(input_dim).to(device)

In [21]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 20
batch_size = 64
learning_rate = 0.001

with mlflow.start_run():

    mlflow.log_param("epochs", epochs)
    mlflow.log_param("batch_size", batch_size)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("optimizer", "Adam")
    mlflow.log_param("loss_function", "MSELoss")

    print("Starting training...")

    for epoch in range(epochs):
        model.train()  # Set model to training mode
        running_loss = 0.0

        for inputs, targets in train_loader:
            # Move data to the device BEFORE the forward pass
            inputs = inputs.to(device)
            targets = targets.to(device)

            # Zero the gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)

            # Calculate loss
            loss = criterion(outputs, targets)

            # Backward pass
            loss.backward()

            # Update weights
            optimizer.step()

            running_loss += loss.item()

        # Calculate the average loss for this epoch
        avg_loss = running_loss / len(train_loader)
        print(
            f"Epoch [{epoch+1}/{epochs}], Training Loss (MSE): {avg_loss:.4f}")

        # 5. THE MISSING PIECE: Log the loss metric to MLflow!
        mlflow.log_metric("train_loss", avg_loss, step=epoch)

# (Your evaluation loop goes down here, also indented under the 'with' block if you want to track it!)

Starting training...
Epoch [1/20], Training Loss (MSE): 55.4924
Epoch [2/20], Training Loss (MSE): 23.3773
Epoch [3/20], Training Loss (MSE): 14.3315
Epoch [4/20], Training Loss (MSE): 10.3461
Epoch [5/20], Training Loss (MSE): 9.5882
Epoch [6/20], Training Loss (MSE): 10.3896
Epoch [7/20], Training Loss (MSE): 8.9703
Epoch [8/20], Training Loss (MSE): 9.1398
Epoch [9/20], Training Loss (MSE): 8.5426
Epoch [10/20], Training Loss (MSE): 9.4066
Epoch [11/20], Training Loss (MSE): 9.4137
Epoch [12/20], Training Loss (MSE): 8.1722
Epoch [13/20], Training Loss (MSE): 7.9097
Epoch [14/20], Training Loss (MSE): 7.3495
Epoch [15/20], Training Loss (MSE): 7.4527
Epoch [16/20], Training Loss (MSE): 7.5032
Epoch [17/20], Training Loss (MSE): 6.9301
Epoch [18/20], Training Loss (MSE): 7.6375
Epoch [19/20], Training Loss (MSE): 6.8770
Epoch [20/20], Training Loss (MSE): 6.7029


In [23]:
mlflow.log_metric("train_loss", avg_loss, step=epoch)

In [24]:
model.eval()
test_loss = 0.0

# Disable gradient calculation for testing
with torch.no_grad():
    for inputs, targets in test_loader:
        # THE FIX: Move both inputs and targets to the device FIRST
        inputs = inputs.to(device)
        targets = targets.to(device)

        # Now the model can process the inputs
        outputs = model(inputs)

        # Now the criterion can compare them, as both are on the GPU
        loss = criterion(outputs, targets)
        test_loss += loss.item()

print(f"\nFinal Test Loss (MSE): {test_loss / len(test_loader):.4f}")


Final Test Loss (MSE): 8.7502


In [25]:
model.eval()

# Create empty lists to store all predictions and true values
all_predictions = []
all_targets = []

# Disable gradient calculation for testing
with torch.no_grad():
    for inputs, targets in test_loader:
        # Move data to the device
        inputs = inputs.to(device)
        targets = targets.to(device)

        # Get model predictions
        outputs = model(inputs)

        # Move outputs and targets back to CPU, convert to numpy, and append to lists
        all_predictions.extend(outputs.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

# Convert the lists into flat NumPy arrays for scikit-learn
y_pred = np.array(all_predictions).flatten()
y_true = np.array(all_targets).flatten()

# Calculate the metrics
mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = root_mean_squared_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print("\n--- Final Model Evaluation ---")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE):  {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R²): {r2:.4f}")


--- Final Model Evaluation ---
Mean Absolute Error (MAE): 1.8763
Mean Squared Error (MSE):  8.7522
Root Mean Squared Error (RMSE): 2.9584
R-squared (R²): 0.9074


In [26]:
mlflow.log_metric("test_mae", mae)
mlflow.log_metric("test_mse", mse)
mlflow.log_metric("test_rmse", rmse)
mlflow.log_metric("test_r2", r2)

In [27]:
mlflow.pytorch.log_model(model, "weather_predictor_model")
print("\nRun successfully logged to MLflow!")

2026/04/19 20:15:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 20:15:54 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/04/19 20:15:55 WARNING mlflow.utils.requirements_utils: Found torch version (2.9.1+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.9.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/19 20:16:01 WARNING mlflow.utils.requirements_utils: Found torch version (2.9.1+cu130) contains a local version label (+


Run successfully logged to MLflow!
